In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
#saved in results\regression-by-source

In [ ]:
# Here I combine datasets with the same source (e.g., ce-*)
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

path = '../datasets/serie-multivariada'
dataframes_by_source = {}

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            #print(f"Processing files with source: {source}")
            df = combine_datasets(path, source)
            
            if df is not None:
                key_name = f"{source.strip('-')}" 
                dataframes_by_source[key_name] = df


In [ ]:
# This function is for predicting the target column values based on the other columns
def predict_and_evaluate_models(dataframe, target, features, source, output_csv):
    df_copy = dataframe.copy()
    normalization_val = df_copy[target].mean()
    X = df_copy[features]
    y = df_copy[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

    models = {
        "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
        "LinearRegression": LinearRegression(),
        "PolynomialRegression": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "AdaBoostRegressor": AdaBoostRegressor(n_estimators=100, random_state=42),
        "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=3),
        "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
        "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42),
        "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    }

    results = {"source": [source]}
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse) / normalization_val

        results[model_name] = [rmse]

    df_results = pd.DataFrame(results)

    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        df_results = pd.concat([existing_df, df_results], ignore_index=True)
    df_results.to_csv(output_csv, index=False)

    return df_results

target_columns = ['Vazao_bbr', 'Vazao_cubic']
features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']  # 'Timestamp_cubic'

for target in target_columns:
    for key, value in dataframes_by_source.items():
        archive = "errorFinding-predict" + target + ".csv"
        source = key
        dataframe = dataframes_by_source[key]
        results = predict_and_evaluate_models(dataframe, target, features, source, archive)


In [ ]:
# Here is the function that randomly removes values from the target column and imputes the data / without normalizing the data for training
def impute_and_evaluate_models(dataframe, target, features, source, output_csv):
    df_copy = dataframe.copy()

    np.random.seed(42)
    missing_rate = 0.2 
    n_missing = int(len(df_copy) * missing_rate)
    missing_indices = np.random.choice(df_copy.index, n_missing, replace=False)
    df_copy.loc[missing_indices, target] = np.nan

    normalization_val = df_copy[target].mean()

    df_known = df_copy.dropna(subset=[target])  
    df_missing = df_copy[df_copy[target].isna()]  

    X_known = df_known[features]
    y_known = df_known[target]

    X_missing = df_missing[features]

    models = {
        "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
        "LinearRegression": LinearRegression(),
        "PolynomialRegression": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "AdaBoostRegressor": AdaBoostRegressor(n_estimators=100, random_state=42),
        "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
        "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
        "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42),
        "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    }

    results = {"source": [source]}
    for model_name, model in models.items():
        model.fit(X_known, y_known)
        imputed = model.predict(X_missing)
        df_copy.loc[df_missing.index, target] = imputed
        original_values = dataframe.loc[missing_indices, target]  # Original values
        imputed_values = df_copy.loc[missing_indices, target]  # Imputed values

        mse = mean_squared_error(original_values, imputed_values)
        rmse = np.sqrt(mse) / normalization_val
        results[model_name] = [rmse]

    df_results = pd.DataFrame(results)

    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        df_results = pd.concat([existing_df, df_results], ignore_index=True)
    df_results.to_csv(output_csv, index=False)

    return df_results

target_columns = ['Vazao_bbr', 'Vazao_cubic']
features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']  # 'Timestamp_cubic'

for target in target_columns:
    for key, value in dataframes_by_source.items():
        archive = "errorFinding-imputation-" + target + ".csv"
        source = key
        dataframe = dataframes_by_source[key]
        results = impute_and_evaluate_models(dataframe, target, features, source, archive)


In [ ]:
# # #aqui eh a funcao que tiro aleatoriamente valores da coluna alvo e imputo os dados / normlizando os dados para treinamento
#esta comandtava pq eu não vi nenhuma vantagem em relação aor mse, entao deixei pra la 

# def imputar_e_avaliar_modelos(dataframe, coluna_alvo, features, source, output_csv="rmse_imputacao_resultados_bbr_minmaxscaler.csv"):
#     df_copy = dataframe.copy()

#     # Introduzir valores NaN aleatórios na coluna alvo
#     np.random.seed(42)
#     missing_rate = 0.2
#     n_missing = int(len(df_copy) * missing_rate)
#     missing_indices = np.random.choice(df_copy.index, n_missing, replace=False)
#     df_copy.loc[missing_indices, coluna_alvo] = np.nan

#     # Normalização para avaliação
#     normalization_val = df_copy[coluna_alvo].mean()

#     # Divisão em conjuntos com e sem valores ausentes
#     df_known = df_copy.dropna(subset=[coluna_alvo])
#     df_missing = df_copy[df_copy[coluna_alvo].isna()]

#     X_known = df_known[features]
#     y_known = df_known[coluna_alvo]

#     X_missing = df_missing[features]

#     # Normalização das features
#     scaler = StandardScaler()
#     X_known = scaler.fit_transform(X_known)
#     X_missing = scaler.transform(X_missing)

#     # Modelos ajustados
#     modelos = {
#         "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
#         "LinearRegression": LinearRegression(),
#         "PolynomialRegression": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
#         "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
#         "AdaBoostRegressor": AdaBoostRegressor(n_estimators=100, random_state=42),
#         "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
#         "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42),
#         "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
#         "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
#         "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
#         "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42),
#         "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1),
#     }

#     resultados = {"source": [source]}
#     for nome_modelo, modelo in modelos.items():
#         modelo.fit(X_known, y_known)
#         imputados = modelo.predict(X_missing)
#         df_copy.loc[df_missing.index, coluna_alvo] = imputados

#         valores_originais = dataframe.loc[missing_indices, coluna_alvo]  # Valores originais
#         valores_imputados = df_copy.loc[missing_indices, coluna_alvo]  # Valores imputados

#         mse = mean_squared_error(valores_originais, valores_imputados)
#         rmse = np.sqrt(mse) / normalization_val
#         resultados[nome_modelo] = [rmse]

#     df_resultados = pd.DataFrame(resultados)

#     if os.path.exists(output_csv):
#         df_existente = pd.read_csv(output_csv)
#         df_resultados = pd.concat([df_existente, df_resultados], ignore_index=True)
#     df_resultados.to_csv(output_csv, index=False)

#     return df_resultados

# # Definição das colunas
# coluna_alvo = 'Vazao_bbr'
# features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']

# # Iteração pelos dataframes
# for key, value in dataframes_por_source.items():
#     source = key
#     dataframe = dataframes_por_source[key]
#     resultados = imputar_e_avaliar_modelos(dataframe, coluna_alvo, features, source)
